In [ ]:
!python --version

In [ ]:
!echo $CONDA_DEFAULT_ENV

## Monai

### Models

  - DenseNet is good, but EfficientNet and ViT outperform it consistently.

### Better

  - EfficientNet-B0/B3 (lightweight, great for cell morphology)
  - ViT-B/16 (large datasets)
  - Swin Transformer (state of the art for microscopy)



In [ ]:
!nvidia-smi

In [ ]:
#!pip3 install opencv-python
#!pip3 instal monai tensorboard

In [ ]:
import os, sys
import numpy as np
import pandas as pd

# from glob import glob
from tqdm import tqdm

import matplotlib.pyplot as plt

from pathlib import Path
ROOT = Path().resolve().parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

# first Basica next pdb_lib
from libs.Basic import *
from libs.image_lib import *


In [ ]:
import torch
from torch.cuda.amp import autocast, GradScaler
from torch.optim import Adam

scaler = GradScaler()
torch.__version__

In [ ]:
torch.cuda.is_available()

In [ ]:
torch.cuda.get_device_name(0)

In [ ]:
import monai
monai.__version__

In [ ]:
from monai.transforms import (
    LoadImage, EnsureChannelFirst, Resize, ScaleIntensity,
    RandFlip, RandRotate, ToTensor, RandGaussianNoise, RandZoom, RandShiftIntensity
)
from monai.data import CacheDataset, DataLoader
from monai.networks.nets import DenseNet121
from monai.losses import DiceCELoss
from monai.utils import set_determinism
from monai.transforms import Compose

In [ ]:
def accuracy(pred, label):
    return (pred.argmax(dim=1) == label).float().mean().item()

In [ ]:
set_determinism(42)

In [ ]:
root0 = "../../colaboracoes/deOcesano/"
os.listdir(root0)

In [ ]:
cp = Cellpose(root0=root0, verbose=True)

In [ ]:
cp.set_default_parameters(root_yaml=os.getcwd(), verbose=True)

In [ ]:
cp.root_samples, cp.root_crop

In [ ]:
plates = cp.list_plates(s_start='Plate')
plates

In [ ]:
i=0
plate=plates[i]

cp.set_plate_params(plate=plate, verbose=True)

In [ ]:
cp.set_plate_params(plate=plate, verbose=False)

ncrop=5

for experiment in cp.experiments:
    cp.create_roots_experiment(experiment, verbose=False)
    fname_imgs = cp.list_crop_images_already_set(ncrop=ncrop, image_type='png', verbose=False)

    key = f"{plate} - {experiment}"
    print(key, len(fname_imgs), cp.root_crop_image)

In [ ]:
filename = os.path.join(cp.root_crop_image, fname_imgs[0])
os.path.exists(filename), filename

### Create dataset dictionary list

In [ ]:
def create_dataset(ncrop:float=5, perc:float=0.6, sel_probes:list=[]):
    dic = {}
    train_list, test_list = [], []
    plates = cp.list_plates(s_start='Plate')

    if perc < 0.2 or perc > .8:
        perc = 0.6

    classes = []
    for plate in plates:
        cp.set_plate_params(plate=plate, verbose=False)

        for experiment in cp.experiments:
            
            mat = experiment.split(' - ')
            probe = mat[0]
            perturb = mat[1]

            if len(sel_probes) > 0 and probe not in sel_probes:
                continue
                
            classes.append(experiment)

    classes = np.unique(classes)
    class_to_index = {_class: i for i, _class in enumerate(classes)}

    icount=-1
    for plate in plates:
        cp.set_plate_params(plate=plate, verbose=False)

        for experiment in cp.experiments:

            mat = experiment.split(' - ')
            probe = mat[0]
            perturb = mat[1]

            if len(sel_probes) > 0 and probe not in sel_probes:
                continue

            cp.create_roots_experiment(experiment, verbose=False)
            fname_imgs = cp.list_crop_images_already_set(ncrop=ncrop, image_type='png', verbose=False)
            fname_imgs = np.array(fname_imgs)

            n = len(fname_imgs)
            lista = np.arange(0, n)
            random.shuffle(lista)
            n_pick = int(n*perc)

            if perc <= 0.30:
                fname_imgs_train = fname_imgs[lista[:n_pick]]
                fname_imgs_test  = fname_imgs[lista[-n_pick:]]
            else:
                fname_imgs_train = fname_imgs[lista[:n_pick]]
                fname_imgs_test  = fname_imgs[lista[n_pick:]]

            key = f"{plate} - {experiment}"
            icount += 1
            dic[icount] = {}
            dic2 = dic[icount]
            dic2['plate_exp'] = key
            dic2['plate'] = plate
            dic2['experiment'] = experiment

            dic2['probe'] = probe
            dic2['perturb'] = perturb
            
            dic2['n'] = len(fname_imgs)
            dic2['n_train'] = len(fname_imgs_train)
            dic2['n_test']  = len(fname_imgs_test)
            dic2['root'] = cp.root_crop_image
            
            for fname in fname_imgs_train:
                train_list.append({"img": os.path.join(cp.root_crop_image, fname), "label": class_to_index[experiment] })

            for fname in fname_imgs_test:
                test_list.append({"img": os.path.join(cp.root_crop_image, fname), "label": class_to_index[experiment] })
                
    df = pd.DataFrame(dic).T
    df = df.sort_values(['probe', 'perturb', 'plate'])
    df = df.reset_index()
    
    return train_list, test_list, classes, df,  class_to_index

In [ ]:
train_list, test_list, classes, dft, class_to_index = create_dataset(ncrop=5, perc=0.6, sel_probes=[])
print(len(dft))
">>> plate_exp_dic", len(dft), len(train_list), ' - first item', train_list[0]

In [ ]:
filename = '../../colaboracoes/deOcesano/crop/Plate1848/FACT - 10%SFB/Overlay_B08_site_11.tif_crop_0_ncrop_5.png'
os.path.exists(filename)

In [ ]:
print(len(dft))
dft

In [ ]:
class_to_index

In [ ]:
len(classes), classes

In [ ]:
train_list[:2]

### Transforms

In [ ]:
# from monai.transforms import RandGaussianNoise, RandZoom, RandShiftIntensity

train_transforms = [
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize((256, 256)),
    RandFlip(spatial_axis=0, prob=0.5),
    RandFlip(spatial_axis=1, prob=0.5),
    RandRotate(range_x=0.1, prob=0.5),
    RandZoom(min_zoom=0.9, max_zoom=1.1, prob=0.3),
    RandGaussianNoise(prob=0.3),
    ScaleIntensity(),    
    ToTensor()
]

In [ ]:
def apply_transforms(sample):
    img = sample["img"]
    label = sample["label"]
    for train_transf_method in train_transforms:
        img = train_transf_method(img)
    return img, torch.tensor(label, dtype=torch.long)

### Dataset + Loader

In [ ]:
def evaluate_loss_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.cuda()
            labels = labels.cuda()
        
            with autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
                preds = outputs.argmax(1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return loss_sum / len(loader), correct / total  

In [ ]:
class CellDataset(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        return apply_transforms(self.items[idx])


### Build data

In [ ]:
try:
    del(train_list)
    del(test_list)
except:
    pass

In [ ]:
sel_probes = ['FACT', 'MMP1', 'Faloidina']
sel_probes = ['FACT']

train_list, test_list, classes, dft, class_to_index = create_dataset(ncrop=5, perc=0.40, sel_probes=sel_probes)
len(train_list), len(test_list), len(classes)

In [ ]:
dft

In [ ]:
classes

In [ ]:
class_to_index

### Model (MONAI DenseNet) – good for biomedical images

In [ ]:
lr=1e-4
weight_decay=1e-4

In [ ]:
model = DenseNet121(
    spatial_dims=2,
    in_channels=3,       # set to 3 if RGB
    out_channels=len(classes),
    pretrained=True
).cuda()

# use Label Smoothing
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
# Try AdamW instead of Adam:
# optimizer = Adam(model.parameters(), lr=lr)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

### Loader

In [ ]:
ds_train = CellDataset(train_list)
train_loader = DataLoader(ds_train, batch_size=16, shuffle=True, num_workers=4)

ds_test = CellDataset(test_list)
test_loader = DataLoader(ds_test, batch_size=16, shuffle=True, num_workers=4)

In [ ]:
train_losses, test_losses, accu_list = [], [], []

In [ ]:
def get_name_model(sel_probes, ncrop):
    if sel_probes == []:
        fname_model = f"cell_model_{ncrop}.pt"
    else:
        fname_model = f"cell_model_{ncrop}_probes_{'_'.join(sel_probes)}.pt"
    
    filename = os.path.join(cp.root_samples, fname_model)
    return filename, fname_model

def save_model(model, optimizer, classes, class_to_index, sel_probes, ncrop, train_losses, test_losses, accu_list):

    filename, fname_model = get_name_model(sel_probes, ncrop)

    try:
        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "classes": classes,
            "class_to_index": class_to_index,
            "train_losses": train_losses,
            "test_losses": test_losses,
            "accu_list": accu_list,
            "n": len(accu_list),
        },  filename)
        print(f"File saved at '{filename}'")
        ret = True
    except:
        print(f"Error: could not save '{filename}'")
        ret = False

    return ret

def read_model(model, optimizer, sel_probes, ncrop):
    filename, fname_model = get_name_model(sel_probes, ncrop)

    if not os.path.exists(filename):
        return False, (model, [], [], [])
    
    new_model = model
    checkpoint = torch.load(filename, map_location="cuda", weights_only=False)

    new_model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    train_losses = checkpoint["train_losses"]
    test_losses = checkpoint["test_losses"]
    accu_list = checkpoint["accu_list"]

    return True, (new_model, train_losses, test_losses, accu_list)


def plot_losses_and_accuracy(train_losses, test_losses, accu_list, figsize=(12,5)):
    epochs_range = range(1, len(train_losses) + 1)
    plt.figure(figsize=figsize)
    
    #------ loss plot ------------------
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, train_losses, marker='o', label='Train Loss')
    plt.plot(epochs_range, test_losses, marker='o', label='Test Loss')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.title("Training & Test Losses")
    
    #----- accuracy plot ---------------
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, accu_list, marker='o', label='Test Accuracy')
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Test Accuracy")
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
filename, fname_model = get_name_model(sel_probes, ncrop)
filename, fname_model

In [ ]:
ret, (new_model, train_losses, test_losses, accu_list) = read_model(model, optimizer, sel_probes, ncrop)

In [ ]:
if ret:
    print("Loading the saved model ...")
    model = new_model
else:
    print("Starting a new model ...")

In [ ]:
len(train_losses), train_losses

In [ ]:
len(test_losses), test_losses

In [ ]:
len(accu_list), accu_list

### Use mixed precision (AMP)

AMP gives faster training + better stability:

In [ ]:
n_epochs = 25
maxi=0

for epoch in range(n_epochs):
    model.train()
    
    tot_train_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}"):
        imgs = imgs.cuda()
        labels = labels.cuda()

        optimizer.zero_grad()

        # with autocast(device_type="cuda", enabled=True, dtype=torch.float16):
        with autocast():
            outputs = model(imgs)
            train_loss = criterion(outputs, labels)
        
        scaler.scale(train_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # train_loss.backward()
        # optimizer.step()
        tot_train_loss += train_loss.item()

    tot_test_loss = 0.0
    correct, tot_labels = 0., 0.
    for imgs, labels in test_loader:
        imgs = imgs.cuda()
        labels = labels.cuda()

        optimizer.zero_grad()
        outputs = model(imgs)
        test_loss = criterion(outputs, labels)

        tot_test_loss += test_loss.item()

        best_pred = outputs.argmax(1)
        correct += (best_pred == labels).sum().item()
        tot_labels += labels.size(0)

    accuracy = correct / tot_labels

    tot_train_loss /= len(train_loader)
    tot_test_loss /= len(test_loader)
    
    train_losses.append(tot_train_loss)
    test_losses.append(tot_test_loss)
    accu_list.append(accuracy)

    if accuracy > maxi:
        maxi = accuracy
        save_model(model, optimizer, classes, class_to_index, sel_probes, ncrop, train_losses, test_losses, accu_list)
        
    print(f"Epoch {epoch+1} - accuracy: {accuracy*100:.1f}% loss train: {tot_train_loss:.4f} test: {tot_test_loss:.4f}")

print("-------- training complete ----------")

In [ ]:
plot_losses_and_accuracy(train_losses, test_losses, accu_list, figsize=(12,5))

In [ ]:
'_'.join(sel_probes)

In [ ]:
if sel_probes == []:
    fname_model = f"cell_model_{ncrop}.pt"
else:
    fname_model = f"cell_model_{ncrop}_probes_{'_'.join(sel_probes)}.pt"


filename = os.path.join(cp.root_samples, fname_model)

try:
    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "classes": classes,
        "class_to_index": class_to_index
    },  filename)
    print(f"File saved at '{filename}'")    
except:
    print(f"Error: could not save '{filename}'")

In [ ]:
epochs_range = range(1, len(train_losses) + 1)
plt.figure(figsize=(12,5))

#------ loss plot ------------------
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, marker='o', label='Train Loss')
plt.plot(epochs_range, test_losses, marker='o', label='Test Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.title("Training & Test Losses")

#----- accuracy plot ---------------
plt.subplot(1, 2, 2)
plt.plot(epochs_range, accu_list, marker='o', label='Test Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Test Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
mean_loss, accuracy = evaluate_loss_accuracy(model, train_loader)
# 20
# (0.9205801197687785, 0.5754133956906631)
# 30
# (0.8070158894062043, 0.6300317354267579)
mean_loss, accuracy

In [ ]:
mean_loss, accuracy = evaluate_loss_accuracy(model, test_loader)
# 20
# (1.0758121476173401, 0.5186236846500751)
# 30
# (1.0233093910217286, 0.5471855687322532)
mean_loss, accuracy